# Exploración y preprocesamiento del corpus de Textos Ordenados del BCRA

Este notebook documenta la exploración de la estructura interna de los 103 Textos Ordenados
del BCRA y el desarrollo de la función de limpieza de texto, como paso previo a la
implementación de la estrategia de chunking por secciones.

**Autora:** Viviana Rodriguez  
**Director:** Ezequiel Nuske  
**Fecha:** Agosto 2026

## Paso 1: Conexión con Google Drive
Se monta Google Drive para acceder al corpus de PDFs descargado en la etapa anterior.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Paso 2: Instalación de dependencias e importación de librerías
Se instala `pypdf`, la librería utilizada para extraer texto de los archivos PDF.

In [13]:
!pip install pypdf --quiet

import pypdf
import re
from pathlib import Path

## Paso 3: Extracción del texto de un PDF de muestra
Se extrae el texto de un documento representativo para analizar su estructura interna. Se observa que los documentos tienen una carátula, un índice con secciones numeradas y un pie de página que se repite en cada página.

In [14]:
pdf_path = Path("/content/drive/MyDrive/tesis-bcra-corpus/t-ctacor.pdf")
reader = pypdf.PdfReader(pdf_path)

for i, page in enumerate(reader.pages[:3]):
    print(f"\n{'='*60}")
    print(f"PÁGINA {i+1}")
    print(f"{'='*60}")
    print(page.extract_text())


PÁGINA 1
CUENTAS DE CORRESPONSALÍA 
-Última comunicación incorporada: “A” 6709-
Texto ordenado al 31/5/209 

PÁGINA 2
B.C.R.A. TEXTO 
ORDENADO DE LAS NORMAS SOBRE 
“CUENTAS DE CORRESPONSALÍA” 
- Índice -
Sección 1. Prestación del servicio de corresponsalía a 
entidades financieras -“su cuenta”-. 
1.1. Entidades intervinientes. 
1.2. Apertura y funcionamiento. Información mínima a requerir. 
1.3. Causales de cierre de cuenta. 
1.4. Entidades financieras del exterior. 
Sección 2. Solicitud del servicio de corresponsalía en 
entidades financieras -“nuestra cuenta”-. 
2.1. Entidades intervinientes. 
2.2. Apertura y 
funcionamiento de cuentas abiertas en entidades financieras del exte-
rior. 
Sección 
3. Solicitud del servicio de corresponsalía por parte de las casas de cambio - “nuestra 
cuenta”. 
3.1. 
Solicitud del servicio en sucursales y subsidiarias de entidades financieras del país 
radicadas en el extranjero. 
3.2. Solicitud del servicio en entidades financieras del exterior. 
Ta

## Paso 4: Verificación de la estructura en múltiples documentos
Se analiza la estructura de tres documentos adicionales para confirmar si el patrón observado se repite.

In [15]:
pdfs = ["t-lavdin.pdf", "t-efemin.pdf", "t-ctacte.pdf"]

for nombre in pdfs:
    pdf_path = Path(f"/content/drive/MyDrive/tesis-bcra-corpus/{nombre}")
    reader = pypdf.PdfReader(pdf_path)
    print(f"\n{'='*60}")
    print(f"ARCHIVO: {nombre}")
    print(f"{'='*60}")
    print(reader.pages[1].extract_text())


ARCHIVO: t-lavdin.pdf
- Índice -
Sección 1. Normas complementarias de prevención del lavado de activos y del financiamiento del 
terrorismo. 
1.1. Normativa aplicable. 
1.2. Información al Banco Central de la República Argentina (BCRA). 
1.3. Procedimientos especiales. 
Sección 2. Pago de cheques y letras de cambio por ventanilla. 
2.1. Limitación. 
2.2. Excepciones. 
2.3. Recaudos. 
Sección 3. Efectivización de créditos en cuentas de depósito. 
3.1. Alcances. 
3.2. Tratamiento específico. 
3.3. Excepciones. 
Sección 4. Disposiciones transitorias. 
Tabla de correlaciones. 
Versión: 11a. COMUNICACIÓN  “A”  6399 V igencia: 
01/01/2018 Página 1 
B.C.R.A.
T
EXTO ORDENADO DE LAS NORMAS SOBRE 
“PREVENCIÓN DEL LAVADO DE ACTIVOS, DEL FINANCIAMIENTO 
DEL TERRORISMO Y OTRAS ACTIVIDADES ILÍCITAS” 

ARCHIVO: t-efemin.pdf
-Índice-
Sección 1. Exigencia.
1.1. Obligaciones comprendidas.
1.2. Base de aplicación.
1.3. Efectivo mínimo.
1.4. Plazo residual.
1.5. Disminución de la exigencia en promedio en

## Paso 5: Detección del patrón de pies de página
Se buscan todas las ocurrencias de la palabra "Sección" en el texto extraído de `t-efemin.pdf` para identificar dónde aparece el pie de página repetitivo.

In [16]:
pdf_path = Path("/content/drive/MyDrive/tesis-bcra-corpus/t-efemin.pdf")
reader = pypdf.PdfReader(pdf_path)

texto_completo = ""
for page in reader.pages:
    texto_completo += page.extract_text() + "\n"

lineas = texto_completo.split('\n')
for i, linea in enumerate(lineas):
    if 'Sección' in linea:
        print(f"Línea {i}: {linea.strip()}")

Línea 4: Sección 1. Exigencia.
Línea 13: Sección 2. Integración.
Línea 20: Sección 3. Incumplimientos.
Línea 24: Sección 4. Base de observancia de las normas.
Línea 26: Sección 5. Responsables y sanciones.
Línea 30: Sección 6. Cupo MiPyME Mínimo.
Línea 31: Sección 7. Disposiciones transitorias.
Línea 74: Sección 1. Exigencia.
Línea 113: Sección 1. Exigencia.
Línea 148: Sección 1. Exigencia.
Línea 194: Sección 1. Exigencia.
Línea 256: Sección 1. Exigencia.
Línea 315: Sección 1. Exigencia.
Línea 358: Sección 1. Exigencia.
Línea 395: Sección 1. Exigencia.
Línea 431: Sección 1. Exigencia.
Línea 468: Sección 1. Exigencia.
Línea 488: Para las entidades financieras alcanzadas por la Sección 6., deberán haber cumplido con
Línea 519: Sección 1. Exigencia.
Línea 562: Sección 1. Exigencia.
Línea 601: Sección 1. Exigencia.
Línea 617: Sección 1. Exigencia.
Línea 668: Sección 2. Integración.
Línea 708: Sección 2. Integración.
Línea 744: Sección 2. Integración.
Línea 772: Sección 3. Incumplimientos.


## Paso 6: Análisis del patrón de pies de página en los 103 documentos
Se ejecuta un análisis automatizado sobre todo el corpus para detectar en cuántos documentos aparece el patrón de pie de página estándar (`B.C.R.A.` seguido de `Sección X.`).

corpus_dir = Path("/content/drive/MyDrive/tesis-bcra-corpus")
pdfs = list(corpus_dir.glob("*.pdf")) + list(corpus_dir.glob("*.PDF"))

patrones_encontrados = {}

for pdf_path in sorted(pdfs):
    try:
        reader = pypdf.PdfReader(pdf_path)
        texto = ""
        for page in reader.pages:
            texto += page.extract_text() + "\n"
        
        lineas = texto.split('\n')
        patrones = []
        for i, linea in enumerate(lineas):
            if 'Sección' in linea and i > 0:
                linea_anterior = lineas[i-1].strip()
                if 'B.C.R.A' in linea_anterior or 'BCRA' in linea_anterior:
                    patrones.append(f"  Línea {i}: {linea.strip()}")
        
        patrones_encontrados[pdf_path.name] = len(patrones)
        if len(patrones) == 0:
            print(f"⚠️  {pdf_path.name}: NO se encontró el patrón esperado")
        else:
            print(f"✅  {pdf_path.name}: {len(patrones)} ocurrencias del patrón")
    
    except Exception as e:
        print(f"❌  {pdf_path.name}: ERROR - {e}")

## Paso 7: Exploración de los documentos sin el patrón estándar
Se analizan los documentos que no siguieron el patrón estándar para identificar si tienen estructuras distintas. Se identifican cuatro grupos:

- **Grupo A:** Secciones numeradas con pie de página estándar (~75 documentos)
- **Grupo B:** Secciones numeradas con pie de página en formato distinto
- **Grupo C:** Organizados por Anexos en lugar de Secciones
- **Grupo D:** Numerados directamente sin la palabra "Sección", o con texto ruidoso

In [17]:
sin_patron = [
    "t-RI2-AE.pdf", "t-RI2-CI.pdf", "t-adrei.pdf", "t-asomut.pdf",
    "t-cedin.pdf", "t-consyr.pdf", "t-convca.pdf", "t-cryl.pdf",
    "t-ctacor.pdf", "t-evacre.pdf", "t-graloc.pdf", "t-jAFIP.pdf",
    "t-micemp.pdf", "t-nmaeef.pdf", "t-nmcief.pdf", "t-rdbcra.pdf",
    "t-reqcac.pdf", "t-retype.pdf", "t-rmgcti.pdf", "t-rmrtsd.pdf",
    "t-seggar.pdf", "t-snp-cec.pdf", "t-snp-tr-nc.pdf", "t-verac.pdf",
    "t-venliq.pdf"
]

for nombre in sin_patron:
    pdf_path = Path(f"/content/drive/MyDrive/tesis-bcra-corpus/{nombre}")
    try:
        reader = pypdf.PdfReader(pdf_path)
        texto = ""
        for page in reader.pages[:2]:
            texto += page.extract_text() + "\n"
        lineas = texto.split('\n')
        print(f"\n{'='*50}")
        print(f"ARCHIVO: {nombre}")
        print(f"{'='*50}")
        for linea in lineas[:20]:
            if linea.strip():
                print(linea.strip())
    except Exception as e:
        print(f"ERROR en {nombre}: {e}")


ARCHIVO: t-RI2-AE.pdf
NORMAS MÍNIMAS SOBRE AUDITORÍAS
EXTERNAS PARA CASAS Y AGENCIAS
DE CAMBIO
-Última comunicación incorporada: “A” 7721-
Texto ordenado al 10/03/2023
Índice
- Anexo I:   Disposiciones generales sobre auditorías externas
- Anexo II: Planeamiento de las auditorías externas
- Anexo III: Procedimientos mínimos de auditoría externa
- Anexo IV: Informes de los auditores externos
B.C.R.A. TEXTO ORDENADO DE LAS NORMAS MÍNIMAS SOBRE AUDITORÍAS EXTERNAS PARA
CASAS Y AGENCIAS DE CAMBIO
Versión: 2a. COMUNICACIÓN  “A”  7721 Vigencia:
11/03/2023 Página 1

ARCHIVO: t-RI2-CI.pdf
NORMAS MÍNIMAS SOBRE CONTROLES
INTERNOS PARA CASAS Y AGENCIAS
DE CAMBIO
-Última comunicación incorporada: “A” 7722-
Texto ordenado al 10/03/2023
NORMAS MÍNIMAS SOBRE CONTROLES INTERNOS PARA CASAS Y AGENCIAS DE
CAMBIO
-Indice-
Sección 1. Aspectos generales.
1.1. Conceptos básicos.
1.2. Disposiciones generales.
Sección 2. Metodología para la evaluación del control interno.
2.1. Ciclos.
2.2. Política de planeam

## Paso 8: Recuperación del archivo corrupto
El archivo `t-capmin.pdf` estaba corrupto por una descarga incompleta. Se elimina y se vuelve a descargar desde el sitio oficial del BCRA.

In [18]:
import urllib.request
import time

url = "https://www.bcra.gob.ar/archivos/Pdfs/Texord/t-capmin.pdf"
destino = Path("/content/drive/MyDrive/tesis-bcra-corpus/t-capmin.pdf")
USER_AGENT = "Mozilla/5.0 (compatible; corpus-downloader/1.0)"

if destino.exists():
    destino.unlink()
    print("Archivo corrupto eliminado.")

for intento in range(4):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
        with urllib.request.urlopen(req, timeout=60) as r:
            data = r.read()
        if not data.startswith(b"%PDF"):
            print("Error: la respuesta no es un PDF válido.")
            break
        destino.write_bytes(data)
        print(f"Descargado correctamente ({len(data)/1024/1024:.2f} MB)")
        break
    except Exception as e:
        print(f"Intento {intento+1} fallido: {e}")
        time.sleep(2 * (intento + 1))

Archivo corrupto eliminado.
Descargado correctamente (5.71 MB)


## Paso 9: Desarrollo de la función de limpieza de pies de página

Se desarrolla la función `limpiar_texto()` que elimina el contenido repetitivo que el PDF inserta en cada página. La función realiza las siguientes limpiezas:

1. Elimina caracteres de control invisibles que pypdf inyecta durante la extracción de texto.
2. Maneja tres variantes del patrón de pie de página detectadas en el corpus:
   - `B.C.R.A.` en una sola línea seguido de `Versión` y `Página X`
   - `B` cortada en una línea y `.C.R.A.` en la siguiente
   - `B.C.R.A` sin punto final, con el nombre del documento en varias líneas
3. Elimina espacios múltiples y líneas vacías excesivas generadas durante la extracción.

In [19]:
def limpiar_texto(texto):
    # Eliminar caracteres de control invisibles
    texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', texto)

    # Unimos la B cortada con el resto de B.C.R.A.
    texto = re.sub(r'B\s*\n\s*\.C\.R\.A\.', 'B.C.R.A.', texto)

    # Eliminamos pies de página con punto final en B.C.R.A.
    texto = re.sub(
        r'B\.C\.R\.A\..*?Página\s*\d+',
        '',
        texto,
        flags=re.DOTALL
    )

    # Eliminamos pies de página sin punto final en B.C.R.A
    texto = re.sub(
        r'B\.C\.R\.A\b.*?Página\s*\d+',
        '',
        texto,
        flags=re.DOTALL
    )

    # Eliminar espacios múltiples y líneas vacías excesivas
    texto = re.sub(r'[ \t]+', ' ', texto)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()

## Paso 10: Verificación de la función de limpieza en los cuatro grupos
Se aplica la función a un documento representativo de cada grupo y se verifican los resultados contando las ocurrencias de "Sección" antes y después de la limpieza.

In [20]:
representantes = {
    "Grupo A (secciones + pie estándar)": "t-efemin.pdf",
    "Grupo B (secciones + pie distinto)": "t-cryl.pdf",
    "Grupo C (organizado por Anexos)": "t-nmaeef.pdf",
    "Grupo D (texto ruidoso)": "t-cedin.pdf"
}

for grupo, nombre in representantes.items():
    pdf_path = Path(f"/content/drive/MyDrive/tesis-bcra-corpus/{nombre}")
    try:
        reader = pypdf.PdfReader(pdf_path)
        texto_crudo = ""
        for page in reader.pages:
            texto_crudo += page.extract_text() + "\n"

        texto_limpio = limpiar_texto(texto_crudo)
        lineas = texto_limpio.split('\n')

        secciones_crudas = sum(1 for l in texto_crudo.split('\n') if 'Sección' in l)
        secciones_limpias = sum(1 for l in lineas if 'Sección' in l)

        print(f"\n{'='*50}")
        print(f"{grupo}: {nombre}")
        print(f"Ocurrencias de 'Sección' antes: {secciones_crudas}")
        print(f"Ocurrencias de 'Sección' después: {secciones_limpias}")
        print(f"Primeras líneas del texto limpio:")
        print('\n'.join(l for l in lineas[:15] if l.strip()))

    except Exception as e:
        print(f"ERROR en {nombre}: {e}")


Grupo A (secciones + pie estándar): t-efemin.pdf
Ocurrencias de 'Sección' antes: 42
Ocurrencias de 'Sección' después: 12
Primeras líneas del texto limpio:
EFECTIVO MÍNIMO
-Última comunicación incorporada: “A” 8443-
Texto ordenado al29/05/26
-Índice-
Sección 1. Exigencia.
1.1. Obligaciones comprendidas.
1.2. Base de aplicación.
1.3. Efectivo mínimo.
1.4. Plazo residual.
1.5. Disminución de la exigencia en promedio en pesos.
1.6. Aumentos puntuales de requerimiento por concentración de pasivos. 
1.7. Traslados.
1.8. Defecto de aplicación de recursos en moneda extranjera.
Sección 2. Integración.
2.1. Conceptos admitidos.

Grupo B (secciones + pie distinto): t-cryl.pdf
Ocurrencias de 'Sección' antes: 43
Ocurrencias de 'Sección' después: 19
Primeras líneas del texto limpio:
“CENTRAL DE REGISTRO Y LIQUIDACIÓN DE 
INSTRUMENTOS DE DEUDA PÚBLICA, 
REGULACIÓN MONETARIA Y FIDEICOMISOS 
FINANCIEROS (CRYL)” 
-Última comunicación incorporada: “A” 8404-
Texto ordenado al 24/02/2026 
-Índice-
Sección